# 技能3 · Day 3 上机：观测数据的因果推断（PSM + IV）

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 在**真实观测数据**（NSW+CPS 观测对照）上用倾向得分匹配（PSM）消除自选择偏差，并与朴素估计对比
2. 在**真实 IV 数据**（close_college 教育回报）上用两阶段最小二乘（2SLS）估计因果效应，解释 LATE
3. 用 DoWhy 完成从建模到估计到反驳检验的完整观测因果分析流程
4. 区分 PSM/DiD/IV/RDD 四大准实验方法的适用条件

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实数据集：NSW+CPS 观测对照（PSM）+ close_college 教育回报（IV）。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml -q

## 1. 数据集背景与营销映射

**两个真实数据集**：
- **NSW+CPS 观测对照**：NSW 实验处理组 + CPS 观测对照组（非随机，严重失衡）--PSM 的用武之地
- **close_college (Card 1995)**：用"是否住近大学"作"受教育年限"的工具变量--IV 最经典真实数据

| 数据变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat`（NSW 培训） | 是否收到优惠券 | 处理 T（PSM） |
| `re78`（1978 收入） | 转化率/GMV | 结果 Y（PSM） |
| `age`,`educ`,`re74`,`re75`,... | 用户画像/历史消费 | 协变量 X（PSM 匹配） |
| `educ`（受教育年限） | 推荐次数/曝光深度 | 内生处理 T（IV） |
| `lwage`（对数工资） | 转化率/GMV | 结果 Y（IV） |
| `nearc4`（住近大学） | 是否有线下门店 | 工具变量 Z（IV） |

**核心问题**：观测对照下朴素估计偏多少？PSM 修正多少？IV 又如何处理未观测混杂？

In [ ]:
import pandas as pd
import numpy as np
import dowhy
from dowhy import CausalModel
from causaldata import nsw_mixtape, cps_mixtape, close_college
import warnings
warnings.filterwarnings('ignore')

## TODO 1-2：加载与探索观测对照数据

In [ ]:
# 1. 加载真实 NSW+CPS 观测对照数据
nsw = nsw_mixtape.load_pandas().data
nsw_treated = nsw[nsw['treat'] == 1]
cps = cps_mixtape.load_pandas().data
cps_similar = cps[(cps['age'] <= 40) & (cps['re75'] <= 10000)]  # 公共支撑限制
df = pd.concat([nsw_treated, cps_similar], ignore_index=True)
print(f"合并后形状: {df.shape}")
print(f"处理组(NSW实验): {(df['treat']==1).sum()}, 对照组(CPS观测): {(df['treat']==0).sum()}")
df.head()

In [ ]:
# 2. 观测对照组 vs 实验处理组的协变量失衡
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
balance = df.groupby('treat')[covariates].mean().T
balance.columns = ['对照组(CPS观测)', '处理组(NSW实验)']
balance['差值'] = balance['处理组(NSW实验)'] - balance['对照组(CPS观测)']
print("协变量均衡性对比（观测对照 vs 实验处理）：")
print(balance)
print()
print("⚠️ 观察：CPS 观测对照组与 NSW 处理组在 age/educ/re75 上严重失衡")
print("   -> 观测对照下朴素估计将有严重自选择偏差，需 PSM 匹配")

## 2. 为什么观测对照下朴素估计严重有偏

Day 1 用 NSW 实验对照（随机化，均衡），后门调整即可。今天换 **CPS 观测对照**--CPS 是全美代表性样本，NSW 是低收入弱势群体，两组协变量**严重失衡**：

- 朴素估计 = ATE + Bias，偏差来自 CPS 对照组与 NSW 处理组的协变量分布不均
- 例如：NSW 处理组 `re75`（前期收入）远低于 CPS 对照组，朴素估计会**高估**培训的正效应

**PSM 的作用**：按倾向得分（收到处理的概率）把处理组与对照组匹配，模拟随机化，消除可观测混杂的自选择偏差。

**注意**：PSM 只消除**可观测**混杂。若有未观测混杂（如"个人上进心"），PSM 仍有偏 -> 此时需 IV（TODO6）。

## 3：朴素估计（观测对照下严重有偏）

In [ ]:
# 3. 朴素估计（观测对照下严重有偏）
naive_ate = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f"朴素估计 ATE = {naive_ate:.2f}")
print("⚠️ 观测对照下严重有偏！CPS 对照组与 NSW 处理组协变量严重失衡")

## TODO 4-5：PSM 倾向得分匹配 + 反驳检验

In [ ]:
# 4. DoWhy + PSM 估计
common_causes = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]
model = CausalModel(data=df, treatment="treat", outcome="re78", common_causes=common_causes)
identified_estimand = model.identify_effect()
print("识别出的估计量：")
print(identified_estimand)
print()
causal_estimate = model.estimate_effect(
    identified_estimand, method_name="backdoor.propensity_score_matching")
print(f"PSM 估计 ATE = {causal_estimate.value:.2f}")
print(f"（对比朴素估计 {naive_ate:.2f}）-- 差异反映自选择偏差大小")

In [ ]:
# 5. 后门回归对比 + 安慰剂反驳
estimate_lr = model.estimate_effect(
    identified_estimand, method_name="backdoor.linear_regression")
print(f"三种估计对比：朴素 {naive_ate:.2f} | 后门回归 {estimate_lr.value:.2f} | PSM {causal_estimate.value:.2f}")
print("解读：朴素估计与后门/PSM 的差异 = 自选择偏差；后门与 PSM 接近程度 = 估计稳健性")
print()
refutation = model.refute_estimate(
    identified_estimand, causal_estimate, "placebo_treatment_refuter")
print(refutation)
print()
print("解读：安慰剂处理下新估计应接近 0 -- 若如此，方法没在虚假处理上'发现'效应，估计可靠。")

## 4. 2026 前沿：双重机器学习（DML）

传统 PSM 用 Logistic 回归估计倾向得分。DML（Chernozhukov 2018, arXiv 1705.07626）用 **ML** 估计 nuisance 参数，通过正交化 + 交叉拟合保持因果可解释性：

1. 用 ML 估计 $E[T|X]$（处理模型）和 $E[Y|X]$（结果模型）
2. 取残差 $\tilde{T} = T - \hat{E}[T|X]$，$\tilde{Y} = Y - \hat{E}[Y|X]$
3. 在残差上回归 $\tilde{Y} \sim \tilde{T}$，得因果效应

实现：`econml.dml` 或 `DoubleML`。DML 放松**函数形式**假设，但不放松**可忽略性**（仍有未观测混杂时仍需 IV）。

## TODO 6：IV 工具变量估计（close_college 教育回报）

In [ ]:
# 6. IV 工具变量估计（close_college 教育回报）
df_iv = close_college.load_pandas().data
print(f"IV 数据形状: {df_iv.shape}")
print(df_iv[['educ', 'lwage', 'nearc4']].head())

model_iv = CausalModel(
    data=df_iv, treatment="educ", outcome="lwage",
    instruments=["nearc4"],
    common_causes=["exper", "black", "smsa", "south", "married"])
identified_iv = model_iv.identify_effect()
estimate_iv = model_iv.estimate_effect(
    identified_iv, method_name="iv.instrumental_variable")

print(f"\nIV 估计（教育对对数工资的因果效应）= {estimate_iv.value:.4f}")
print("解读：这是 LATE（局部平均处理效应）-- 对 compliers（因 nearc4 变化而改变 educ 的人）有效。")
print("若 IV 估计 > OLS 估计，暗示 OLS 存在向下偏差（如能力偏误：能力高的人既多上学又高工资）。")

## 5. 反思与前沿

### 反思问题
1. PSM 估计 vs 朴素估计的差异来自哪些可观测混杂？（看 TODO2 的失衡表，哪个协变量差距最大）
2. PSM 与后门回归估计接近吗？接近说明什么？不接近呢？
3. 安慰剂检验结果是否支持 PSM 估计的稳健性？
4. IV 估计 vs OLS 估计的差异说明了什么？若 IV > OLS，是否暗示 OLS 存在能力偏误（向下偏差）？
5. IV 估计的是 LATE 而非 ATE--这对营销应用（如"用门店作推荐次数的工具变量"）有什么启示？

### 2026 前沿：双重机器学习（DML）
DML 用 ML 估计 nuisance 参数 + 正交化 + 交叉拟合，放松函数形式假设，2026 年仍是观测因果前沿。在数字广告归因、价格弹性估计等营销场景被越来越多采用。
参考 arXiv 1705.07626（Chernozhukov et al. 2018）。**注意**：DML 不放松可忽略性假设，有未观测混杂仍需 IV。

> 🔗 深入阅读见 `reading.md` 的 DML 条目。